# 02. Pra-pemrosesan Teks Arab dan Pembagian Dataset

Notebook ini menjalankan **Tahap 2 metodologi penelitian** dengan sumber eksklusif `data/basic_articles_dataset.csv` dan `data/stopwords.txt`:
1. **Pembersihan Teks (`preprocess_text`)**: konversi huruf kecil (*lowercasing*), penghapusan angka (`\d+`), penghapusan tanda baca Arab & ASCII (`،؛؟”0123456789“` dan `string.punctuation`), serta tokenisasi berbasis spasi.
2. **Percabangan *Stopword Removal***:
   - **Medan Makna (*Unigram*)**: menghapus *stopwords* gabungan `data/stopwords.txt` dan `Arabic-Stopwords`.
   - **Semantik *Frame* (*Bigram & Trigram*)**: mempertahankan kata tugas/preposisi Arab (`في`, `من`, `على`, `إلى`, `أن`) sebagai pengikat konteks 2–3 kata.
3. **Pembagian Dataset Deterministik (`RANDOM_SEED = 42`)**:
   - Skema **Medan Makna (`80% / 15% / 5%`)** melalui `split_dataset_80_15_5()`.
   - Skema **Semantik *Frame* (`85% / 10% / 5%`)** melalui `split_dataset_85_10_5()`, sekaligus memilih **11 artikel uji representatif** (1 per kategori) dari split uji `basic_articles_dataset.csv` dan menyimpannya ke `results/stats/sample_11_test_articles.csv`.


In [1]:
import sys
from pathlib import Path

# Resolve root repositori secara relatif
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "src" / "arabic_news").exists() else cwd.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {REPO_ROOT.name}")

import json
import pandas as pd
from IPython.display import display

from arabic_news.config import (
    LABELS,
    LABELS_ID,
    RANDOM_SEED,
    STATS_DIR,
    resolve_data_file,
)
from arabic_news.evaluation import (
    select_one_sample_per_category,
    split_dataset_80_15_5,
    split_dataset_85_10_5,
)
from arabic_news.preprocessing import (
    ARABIC_PUNCTUATION,
    load_stopwords,
    preprocess_text,
)


Repository root: arabic_news_classification


## 2.1 Muat `data/stopwords.txt` & Gabungan Kata Henti Arab


In [2]:
stopwords_path = resolve_data_file("stopwords.txt")
if not stopwords_path.exists():
    raise FileNotFoundError("Letakkan stopwords.txt di folder data/ sebelum menjalankan notebook ini.")

stopwords_list = load_stopwords(stopwords_path=stopwords_path, include_library_stopwords=True)
print(f"Karakter tanda baca & angka yang dibersihkan : {ARABIC_PUNCTUATION}")
print(f"Jumlah kata henti gabungan                   : {len(stopwords_list):,} kata")
print(f"25 contoh pertama kata henti                 : {stopwords_list[:25]}")


Karakter tanda baca & angka yang dibersihkan : ،؛؟”0123456789“!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
Jumlah kata henti gabungan                   : 13,911 kata
25 contoh pertama kata henti                 : ['أن', 'أنك', 'أنكما', 'أنكم', 'أنكن', 'أننا', 'أنها', 'أنه', 'أنهما', 'أنهم', 'أنهن', 'أني', 'بأن', 'بأنك', 'بأنكما', 'بأنكم', 'بأنكن', 'بأننا', 'بأنها', 'بأنه', 'بأنهما', 'بأنهم', 'بأنهن', 'بأني', 'فأن']


## 2.2 Pembagian Dataset dari `data/basic_articles_dataset.csv` & Pemilihan 11 Sampel Uji

Membagi `basic_articles_dataset.csv` untuk kedua skema penelitian, mengambil 1 artikel per kategori dari partisi uji (*test split*), menyimpannya ke `results/stats/sample_11_test_articles.csv`, lalu membandingkan hasil pra-pemrosesan tanpa vs dengan *stopword removal* pada ke-11 artikel uji tersebut.


In [3]:
basic_csv = resolve_data_file("basic_articles_dataset.csv")
df_basic = pd.read_csv(basic_csv, low_memory=False)

# Split 80/15/5 untuk Medan Makna
tr_m, val_m, te_m, y_tr_m, y_val_m, y_te_m = split_dataset_80_15_5(
    df_basic, random_state=RANDOM_SEED, shuffle_first=True
)
print(
    f"[Medan Makna 80/15/5] Total valid={len(tr_m)+len(val_m)+len(te_m):,} | "
    f"Train={len(tr_m):,} | Val={len(val_m):,} | Test={len(te_m):,}"
)

# Split 85/10/5 untuk Semantik Frame
tr_f_df, val_f_df, te_f_df = split_dataset_85_10_5(df_basic, random_state=RANDOM_SEED)
print(
    f"[Frame Semantik 85/10/5] Total valid={len(tr_f_df)+len(val_f_df)+len(te_f_df):,} | "
    f"Train={len(tr_f_df):,} | Val={len(val_f_df):,} | Test={len(te_f_df):,}"
)

# Ambil 11 artikel sampel uji nyata dari test split basic_articles_dataset.csv
sample_11_df = select_one_sample_per_category(te_f_df, random_state=RANDOM_SEED)

STATS_DIR.mkdir(parents=True, exist_ok=True)
sample_out = STATS_DIR / "sample_11_test_articles.csv"
sample_11_df.to_csv(sample_out, index=False, encoding="utf-8")

split_cat_df = pd.DataFrame({
    "Label": list(LABELS.keys()),
    "Kategori": list(LABELS.values()),
    "Frame Train (85%)": [int((tr_f_df["label"] == cid).sum()) for cid in LABELS],
    "Frame Val (10%)": [int((val_f_df["label"] == cid).sum()) for cid in LABELS],
    "Frame Test (5%)": [int((te_f_df["label"] == cid).sum()) for cid in LABELS],
    "Medan Val (15%)": [int((y_val_m == cid).sum()) for cid in LABELS],
})
(STATS_DIR / "split_summary.json").write_text(
    json.dumps(
        {
            "medan_makna_80_15_5": {"train": len(tr_m), "val": len(val_m), "test": len(te_m)},
            "frame_semantik_85_10_5": {"train": len(tr_f_df), "val": len(val_f_df), "test": len(te_f_df)},
            "per_category": split_cat_df.to_dict(orient="records"),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
display(split_cat_df)
print(f"11 artikel uji tersimpan ke {sample_out.relative_to(REPO_ROOT)}")


[Medan Makna 80/15/5] Total valid=170,934 | Train=136,746 | Val=25,641 | Test=8,547
[Frame Semantik 85/10/5] Total valid=170,934 | Train=145,293 | Val=17,094 | Test=8,547


,Label,Kategori,Frame Train (85%),Frame Val (10%),Frame Test (5%),Medan Val (15%)
0,1,أمن,14740,1734,867,2702
1,2,اقتصاد,13514,1590,795,2409
2,3,تكنولوجيا,9381,1103,552,1629
3,4,ثقافة,9217,1085,542,1576
4,5,دوليات,15269,1797,898,2680
5,6,رياضة,15820,1861,931,2807
6,7,سياحة,15512,1826,912,2637
7,8,سياسة,15551,1829,915,2807
8,9,صحة,11271,1326,663,1971
9,10,علم,12069,1420,710,2127


11 artikel uji tersimpan ke results/stats/sample_11_test_articles.csv


## 2.3 Perbandingan Hasil Pra-pemrosesan (Retensi vs Penghapusan *Stopwords*) pada 11 Artikel Uji


In [4]:
stop_set = set(stopwords_list)
prep_rows = []
for _, row in sample_11_df.iterrows():
    cid = int(row["label"])
    raw_txt = str(row["text"])
    clean_frame = preprocess_text(raw_txt, stopwords=None)
    clean_medan = preprocess_text(raw_txt, stopwords=stop_set)
    prep_rows.append({
        "Label": cid,
        "Kategori": f"{LABELS[cid]} ({LABELS_ID[cid]})",
        "Sumber": row["source"],
        "Kata Mentah": len(raw_txt.split()),
        "Token Semantik Frame (Utuh)": len(clean_frame.split()),
        "Token Medan Makna (Tanpa Stopwords)": len(clean_medan.split()),
        "Cuplikan Bersih (Frame)": " ".join(clean_frame.split()[:8]) + " ...",
    })

display(pd.DataFrame(prep_rows))


,Label,Kategori,Sumber,Kata Mentah,Token Semantik Frame (Utuh),Token Medan Makna (Tanpa Stopwords),Cuplikan Bersih (Frame)
0,1,أمن (Keamanan),alsumaria,224,220,161,مؤتمر تعقده الكنائس المشرقية في لبنان حول الوج...
1,2,اقتصاد (Ekonomi),saraya,587,554,402,طلب مرتفع على القرطاسية والزي مع بدء العام ...
2,3,تكنولوجيا (Teknologi),alsharq,380,374,262,مايكروسوفت تعلن عن تطبيق جديد للدخول لحساباتك ...
3,4,ثقافة (Budaya),assahraa,21,21,19,غروتوفسكي يغادر تاركا وراءه مسرحه الفقير وروحه...
4,5,دوليات (Internasional),alsharq,225,225,166,انفجارات قوية تهز العاصمة الأوكرانية كييفهزت ا...
5,6,رياضة (Olahraga),alsharq,290,282,183,إيقاف آدم صاعقة تقع على الخريطياتقوبل قرار الإ...
6,7,سياحة (Travel / Pariwisata),akhbarelsyaha,737,720,529,المطارات العالمي يجدد شهادة اعتماد كربوني لمطا...
7,8,سياسة (Politik),alsumaria,48,48,42,انتخاب الحكيم رئيسا لتحالف عراقيوناعلن تحالف ع...
8,9,صحة (Kesehatan),addustour,288,271,177,كورونا الإصابات مليونا والوفيات ملايينعواصم ار...
9,10,علم (Sains),alsumaria,163,163,110,كيف تتخلص من صداع أولى أيام رمضانيختلف نظام ...
